# Supplementary figures — the `Fig2.ipynb` group

Ten supplementary panels cleaned out of `../Fig2.ipynb`, which also produces
main Figure 2.

| Panel | What it shows |
|---|---|
| `sup_fig4A` | median unproductive splicing vs RIN, per tissue |
| `sup_fig4B` | ... vs logUPF3A |
| `sup_fig4UPF1` / `4UPF2` / `4UPF3B` | ... vs logUPF1 / logUPF2 / logUPF3B |
| `sup_fig5A` / `5B` | the same relationship per tissue, 7 × 7 grid (logUPF3A / RIN) |
| `sup_fig7A` / `7B` / `7C` | Fig. 2e boxplot for MYOM2 / SRSF3 / DLG4 |

Panels are written to `plots/`. See `README.md` for the full supplementary map.

**Reuse:** the data chain is shared with main Figure 2, so this imports
`../Figure2/Figure2_helpers.py` rather than duplicating the GTEx TPM reader and
the unproductive-percentage loader. `sup_fig7A/B/C` reuse
`Figure2_plot_helpers.plot_gene_boxplots` directly — they are Fig. 2e for three
other genes.

**Two quirks preserved from the original**, both documented at the cells:
the subsample band uses 15 tissues for `sup_fig4A` but 30 for the gene panels,
and the band was unseeded (so it moved between runs) — `seed=0` here.

In [ ]:
import importlib
import os

from matplotlib import pyplot as plt

import SupFig_Fig2_plot_helpers
import SupFig_Fig2_helpers
importlib.reload(SupFig_Fig2_plot_helpers)
importlib.reload(SupFig_Fig2_helpers)

from SupFig_Fig2_helpers import (build_sup_upf_df, make_median_by_tissue,
                                fit_rlm_with_band, make_per_tissue_series,
                                make_boxplot_df, PLOTS_DIR,
                                TEN_TISSUES, TEN_TISSUES_CLEAN, BOXPLOT_PALETTE)
from SupFig_Fig2_plot_helpers import plot_median_scatter, plot_per_tissue_grid

# sup_fig7A/B/C are the Figure 2e boxplot for three other genes, so they reuse
# that plot helper directly instead of a near-duplicate here.
import sys
sys.path.insert(0, os.path.join('..', 'Figure2'))
from Figure2_plot_helpers import plot_gene_boxplots

os.makedirs(PLOTS_DIR, exist_ok=True)
print('panels ->', PLOTS_DIR)


def save(name, out_dir, dpi=300):
    """Write one panel as png + pdf + svg, as the source notebooks did."""
    for ext in ('png', 'pdf', 'svg'):
        plt.savefig(f'{out_dir}/{name}.{ext}', dpi=dpi, bbox_inches='tight')
    print('wrote', f'{out_dir}/{name}.[png|pdf|svg]')

In [ ]:
# Fast path: re-plot from the pickles written by the cell below.
# Uncomment this cell and skip the next one.

# import pickle
# with open('figure_data/fig2group_upf_df.pickle', 'rb') as fh:
#     upf_df = pickle.load(fh)

In [ ]:
# Fig2-derived data. Reads the GTEx TPM table (15 genes), the leafcutter2
# unproductive-percentage tables and the GTEx sample attributes for RIN.
# This is the heavy cell for every sup_fig4*/5*/7* panel below.

import pickle

upf_df = build_sup_upf_df()
os.makedirs('figure_data', exist_ok=True)
with open('figure_data/fig2group_upf_df.pickle', 'wb') as fh:
    pickle.dump(upf_df, fh)

print(upf_df.shape, '->', [c for c in upf_df.columns if c in ('pct', 'RIN', 'logUPF3A')])

## `sup_fig4*` — median unproductive splicing vs RIN and the UPF genes

Five panels, one function: per-tissue medians with a robust linear fit and a
subsample band. Only the x variable, the axis label and the annotation differ.

Note the band's subsample size differs between them in `Fig2.ipynb`: 15 tissues
for `sup_fig4A` (RIN), 30 for the four gene panels. The originals left the RNG
unseeded, so the band varied run to run; `seed=0` here makes it reproducible.

In [ ]:
# sup_fig4A -- from Fig2.ipynb (RIN)
median_rin = make_median_by_tissue(upf_df, 'RIN')
fit_rin = fit_rlm_with_band(median_rin['data'], 'RIN', draw_size=15, seed=0)

plot_median_scatter(median_rin, fit_rin, xlabel='RIN')
save('sup_fig4A', PLOTS_DIR)

# Original: plt.savefig('../code/plots/sup_fig4A.png', dpi=300, bbox_inches='tight')

In [ ]:
# sup_fig4B / sup_fig4UPF1 / sup_fig4UPF2 / sup_fig4UPF3B -- from Fig2.ipynb
# The rho / p-value annotations are as hand-written in the source notebook.
import numpy as np

GENE_PANELS = [
    ('sup_fig4B',      'logUPF3A', 'logUPF3A (TPM)',
     r'$\rho=0.62$' + '\np-val ' + r'$\leq 2.2e^{-6}$', (np.log10(60), 0.5)),
    ('sup_fig4UPF1',   'logUPF1',  'logUPF1 (TPM)',  None, None),
    ('sup_fig4UPF2',   'logUPF2',  'logUPF2 (TPM)',  None, None),
    ('sup_fig4UPF3B',  'logUPF3B', 'logUPF3B (TPM)', None, None),
]

for name, xvar, xlabel, annot, annot_xy in GENE_PANELS:
    md_ = make_median_by_tissue(upf_df, xvar)
    fit = fit_rlm_with_band(md_['data'], xvar, draw_size=30, seed=0)
    plot_median_scatter(md_, fit, xlabel=xlabel, annotation=annot, annotation_xy=annot_xy)
    save(name, PLOTS_DIR)
    plt.close('all')

## `sup_fig5A` / `sup_fig5B` — the same relationship per tissue

7 × 7 grid, one cell per GTEx tissue, Spearman rho and p annotated in each.
Artery-Tibial gets a black fit line; the rest red. The 49th cell is blank.

In [ ]:
# sup_fig5A -- from Fig2.ipynb (logUPF3A, per tissue)
series_upf3a = make_per_tissue_series(upf_df, 'logUPF3A', seed=0)
plot_per_tissue_grid(series_upf3a, xlabel='logUPF3A (TPM)')
save('sup_fig5A', PLOTS_DIR)
plt.close('all')

In [ ]:
# sup_fig5B -- from Fig2.ipynb (RIN, per tissue)
series_rin = make_per_tissue_series(upf_df, 'RIN', seed=0)
plot_per_tissue_grid(series_rin, xlabel='RIN')
save('sup_fig5B', PLOTS_DIR)
plt.close('all')

## `sup_fig7A/B/C` — the Figure 2e boxplot for three more genes

Identical to Fig. 2e (`GABBR1`) but for MYOM2, SRSF3 and DLG4, so these reuse
`Figure2_plot_helpers.plot_gene_boxplots` directly.

In [ ]:
# sup_fig7A/B/C -- from Fig2.ipynb (MYOM2, SRSF3, DLG4)
for name, gene in [('sup_fig7A', 'MYOM2'), ('sup_fig7B', 'SRSF3'), ('sup_fig7C', 'DLG4')]:
    bdf = make_boxplot_df(upf_df, gene=gene, tissues=TEN_TISSUES)
    plot_gene_boxplots(bdf, gene, TEN_TISSUES_CLEAN, BOXPLOT_PALETTE)
    save(name, PLOTS_DIR)
    plt.close('all')